In [1]:
import uuid
from typing import Optional

import polars as pl
import sqlalchemy as sa
from faker import Faker
from IPython.display import display
from pydantic import BaseModel

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)
fake = Faker('es_MX')


class Area(BaseModel):
    id_area: int
    nombre_area: str


class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str


class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str


class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None


class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1


def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]


def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(
            id_habilidad=row['id_habilidad'],
            descripcion=row['descripcion']
        )
        for row in rows
    ]


def generate_name_and_email() -> tuple[str, str]:
    name = fake.name()
    fake_email = fake.email()
    email_local, email_domain = fake_email.split('@', 1)
    email = f'{email_local}_{uuid.uuid4().hex[:10]}@{email_domain}'
    return name, email


def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )


def submit_user_habilities(
    id_usuario: int,
    habilidad_ids: list[int]
) -> None:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            DELETE FROM usuario_habilidad
            WHERE id_usuario = :id_usuario;
        '''), {'id_usuario': id_usuario})
        if not habilidad_ids:
            return
        conn.execute(sa.text('''
            INSERT INTO usuario_habilidad
                (id_usuario, id_habilidad, cumplimiento_criterio)
            VALUES (:id_usuario, :id_habilidad, 1.0);
        '''), [
            {'id_usuario': id_usuario, 'id_habilidad': id_habilidad}
            for id_habilidad in habilidad_ids
        ])


def construct_result(
    uuid_usuario: str,
    id_carrera: int,
    porcentaje_coincidencia: float = 1.0,
    top: int = 1
) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )


def run_happy_path() -> pl.DataFrame:
    areas = get_all_areas()
    for area in areas:
        area_select = Area(
            id_area=area['id_area'],
            nombre_area=area['nombre_area']
        )
        carreras = get_careers_by_area_select(area_select.id_area)

        for carrera in carreras:
            nombre, correo = generate_name_and_email()
            user = Usuario(
                uuid_usuario=f'USER-{uuid.uuid4()}',
                nombre=nombre,
                correo=correo,
                id_area=area_select.id_area
            )
            user_db = submit_user(user)
            habilidades = get_habilities_by_career(carrera.id_carrera)
            habilidad_ids = [
                habilidad.id_habilidad
                for habilidad in habilidades
            ]
            submit_user_habilities(user_db.id_usuario, habilidad_ids)
            construct_result(user_db.uuid_usuario, carrera.id_carrera)

    query = '''
        SELECT
            u.id_usuario,
            u.nombre AS nombre_usuario,
            u.uuid_usuario,
            c.id_carrera,
            c.nombre_carrera AS carrera,
            r.porcentaje_coincidencia,
            h.id_habilidad,
            h.descripcion AS habilidad,
            uh.cumplimiento_criterio
        FROM resultado r
        JOIN usuario u ON u.uuid_usuario = r.uuid_usuario
        JOIN carrera c ON c.id_carrera = r.id_carrera
        JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
        JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
        JOIN usuario_habilidad uh
            ON uh.id_usuario = u.id_usuario
            AND uh.id_habilidad = h.id_habilidad
        ORDER BY c.id_carrera, u.id_usuario, h.id_habilidad;
    '''

    with engine.begin() as conn:
        return pl.read_database(query, connection=conn)


if __name__ == '__main__':
    resultados_df = run_happy_path()
    print(f'Carreras procesadas: {len(get_all_careers())}')
    print(f'Filas de habilidades/resultados: {resultados_df.height}')
    display(resultados_df)

Carreras procesadas: 66
Filas de habilidades/resultados: 336


id_usuario,nombre_usuario,uuid_usuario,id_carrera,carrera,porcentaje_coincidencia,id_habilidad,habilidad,cumplimiento_criterio
i64,str,str,i64,str,"decimal[38,2]",i64,str,f64
1,"""Rolando Genaro Segura Perea""","""USER-6cd5bff9-11ca-4897-b6db-5…",1,"""Ingeniería Aeronáutica""",1.00,1,"""Diseño, construcción, mantenim…",1.0
1,"""Rolando Genaro Segura Perea""","""USER-6cd5bff9-11ca-4897-b6db-5…",1,"""Ingeniería Aeronáutica""",1.00,2,"""Proyección y puesta en operaci…",1.0
1,"""Rolando Genaro Segura Perea""","""USER-6cd5bff9-11ca-4897-b6db-5…",1,"""Ingeniería Aeronáutica""",1.00,3,"""Investigación, adaptación y de…",1.0
1,"""Rolando Genaro Segura Perea""","""USER-6cd5bff9-11ca-4897-b6db-5…",1,"""Ingeniería Aeronáutica""",1.00,4,"""Planeación, dirección y gestió…",1.0
1,"""Rolando Genaro Segura Perea""","""USER-6cd5bff9-11ca-4897-b6db-5…",1,"""Ingeniería Aeronáutica""",1.00,5,"""Aplicación de normatividad téc…",1.0
…,…,…,…,…,…,…,…,…
66,"""Aida Victoria Sauceda""","""USER-2a878df4-1d96-4fc9-aab6-3…",66,"""Licenciatura en Turismo""",1.00,23,"""Liderazgo, ética profesional y…",1.0
66,"""Aida Victoria Sauceda""","""USER-2a878df4-1d96-4fc9-aab6-3…",66,"""Licenciatura en Turismo""",1.00,250,"""Planeación, dirección y gestió…",1.0
66,"""Aida Victoria Sauceda""","""USER-2a878df4-1d96-4fc9-aab6-3…",66,"""Licenciatura en Turismo""",1.00,251,"""Diseño, desarrollo y comercial…",1.0
